# Emergency Department Triage: Acuity Prediction Model
## Predicting Patient Acuity Level (ESI 1-5) from Clinical Features

**Objective**: Build a machine learning model to predict patient acuity level (ESI 1-5) using clinical data and chief complaints.

**Model Architecture**:
- **LightGBM Classifier** for multi-class ordinal classification
- **ClinicalBERT embeddings** for semantic understanding of clinical text (chief complaints)
- **Chi-Square TF-IDF feature selection** for extracting discriminative clinical keywords
- **Advanced feature engineering** (BP ratios, cyclic encoding, comorbidity interactions)

**Key Features**:
- ClinicalBert for clinical text embeddings from chief_complaint_raw
- Chi2TextFeatureSelector for TF-IDF feature selection per class
- Advanced feature engineering (cyclic encoding, BP ratios, comorbidity interactions)
- Weighted Kappa metrics for ordinal classification (clinically meaningful)
- SHAP explainability for clinical trust and interpretability

**Evaluation Metrics**:
- Accuracy, Cohen's Kappa, Confusion Matrix
- Undertriage/Overtriage analysis for clinical safety
- Per-class performance (Precision, Recall, F1-Score)

**Date**: 2026-04-04


In [ ]:
# ============================================================================
# SECTION 1: IMPORTS - Core Libraries for Acuity Prediction Pipeline
# ============================================================================
# This notebook implements a single-stage acuity prediction pipeline:
# INPUT: Clinical data (vitals, demographics, chief complaints)
# OUTPUT: Predicted acuity level (ESI 1-5)
#
# Libraries imported:
# - Data manipulation: pandas, numpy
# - ML & preprocessing: scikit-learn pipelines, transformers, model selection
# - Gradient boosting: LightGBM (primary model for acuity classification)
# - NLP: HuggingFace Transformers (ClinicalBERT for clinical text embeddings)
# - Visualization: matplotlib, seaborn
# - Text processing: TfidfVectorizer, chi2 feature selection
# - Model evaluation: confusion matrix, f1-score, cohen_kappa_score (ordinal metrics)

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn: preprocessing, pipelines, and model selection
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, 
    cohen_kappa_score, 
    confusion_matrix, 
    classification_report,
    f1_score,
    roc_auc_score
)

# Ensemble and gradient boosting models
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from lightgbm import LGBMClassifier

# Text processing for chief complaint narratives
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import chi2
from sklearn.base import BaseEstimator, TransformerMixin

# Transformers library: state-of-the-art NLP models for clinical text
try:
    import torch
    from transformers import AutoTokenizer, AutoModel
    TRANSFORMERS_AVAILABLE = True
    DEVICE = torch.device('xpu' if torch.xpu.is_available() else 'cpu')
except ImportError:
    print("⚠️  HuggingFace transformers not installed. Will use TF-IDF fallback.")
    TRANSFORMERS_AVAILABLE = False
    DEVICE = 'cpu'

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seeds for reproducibility across runs
np.random.seed(42)

print("✓ All imports successful!")
print(f"✓ Transformers available: {TRANSFORMERS_AVAILABLE}")
print(f"✓ Device for embeddings: {DEVICE}")

# SECTION 2: DATA LOADING AND EXPLORATION

## Step 1: Load and Merge Raw CSV Files
We load clinical data from four CSV files and merge them on `patient_id`:
- `train.csv`: Contains triage_acuity (target ESI 1-5), disposition, ed_los_hours
- `test.csv`: Test set (no labels)
- `chief_complaints.csv`: Raw text chief complaints + system category
- `patient_history.csv`: Medical history and comorbidities

**Data Preprocessing**:
- Merge patient information (complaints + medical history)
- **Drop 'chief_complaint_system'** to prevent information leakage (categorical features derived from text)
- Keep only raw text ('chief_complaint_raw') for feature extraction
- Ensure all unique patients are captured


In [ ]:
# ============================================================================
# SECTION 2: DATA LOADING AND MERGING
# ============================================================================
# Goal: Load clinical data from multiple CSV files and merge on patient_id
# Target: triage_acuity (ESI 1-5 rating)
# Features: Vitals (HR, BP, RR, SpO2), demographics, medical history, text

print("=" * 80)
print("LOADING DATA FILES")
print("=" * 80)

# Load train and test sets
train_df = pd.read_csv("dataset/train.csv")
test_df = pd.read_csv("dataset/test.csv")

# Load patient information (chief complaints and medical history)
chief_complaints_df = pd.read_csv("dataset/chief_complaints.csv")
patient_history_df = pd.read_csv("dataset/patient_history.csv")

print(f"\n📊 INITIAL DATASET SHAPES:")
print(f"   train.csv:                {train_df.shape}")
print(f"   test.csv:                 {test_df.shape}")
print(f"   chief_complaints.csv:     {chief_complaints_df.shape}")
print(f"   patient_history.csv:      {patient_history_df.shape}")

# ============================================================================
# MERGE: Combine chief_complaints + patient_history on patient_id
# ============================================================================
# Use 'outer' join to capture all unique patients. NaN values indicate
# missing info for a particular patient record.

patient_info_df = chief_complaints_df.merge(
    patient_history_df,
    on="patient_id",
    how="outer"
)

print(f"\n📋 MERGED PATIENT INFO: {patient_info_df.shape}")

# ============================================================================
# DROP LEAK COLUMNS
# ============================================================================
# CRITICAL: 'chief_complaint_system' is derived from 'chief_complaint_raw'
# Using it would create information leakage (model shortcuts through category).
# We keep ONLY raw text for NLP feature extraction.

if 'chief_complaint_system' in train_df.columns:
    train_df.drop(columns=['chief_complaint_system'], inplace=True)
if 'chief_complaint_system' in test_df.columns:
    test_df.drop(columns=['chief_complaint_system'], inplace=True)

print("✓ Dropped 'chief_complaint_system' to prevent data leakage")

# ============================================================================
# MERGE: Add patient info to train and test sets
# ============================================================================
# Left join: Keep all training rows, add matching patient info

train_full_df = train_df.merge(
    patient_info_df,
    on="patient_id",
    how="left"
)

test_full_df = test_df.merge(
    patient_info_df,
    on="patient_id",
    how="left"
)

print(f"\n✓ MERGED TRAIN+INFO: {train_full_df.shape}")
print(f"✓ MERGED TEST+INFO:  {test_full_df.shape}")

# Display basic sample
print(f"\n📌 SAMPLE ROW (first row of train_full_df):")
print(train_full_df.iloc[0][:10])  # Show first 10 columns

In [ ]:
# ============================================================================
# PREPARE DATA FOR ACUITY PREDICTION
# ============================================================================

# Numeric mapping for disposition (for reference, not used in this stage)
DISPOSITION_DICT = {
    'discharged': 0,    # Patient released from ED (lowest medical complexity)
    'admitted': 1,      # Patient admitted to hospital ward (high acuity indicator)
    'transferred': 2,   # Patient transferred to another facility
    'observation': 3,   # Patient placed under observation
    'lwbs': 4,          # Left Without Being Seen (administrative)
    'lama': 5,          # Left Against Medical Advice (patient choice)
    'deceased': 6       # Patient died in ED (highest severity indicator)
}

# Reverse mapping for later decoding
NUM_TO_DISPOSITION = {v: k for k, v in DISPOSITION_DICT.items()}

# Apply mapping to disposition (reference only - acuity is the main target)
train_full_df['disposition'] = train_full_df['disposition'].map(DISPOSITION_DICT)

print(f"\n🔢 DISPOSITION MAPPING (for reference):")
for k, v in DISPOSITION_DICT.items():
    print(f"   {v}: {k}")

# ============================================================================
# TARGET AND FEATURES FOR ACUITY PREDICTION
# ============================================================================
# PRIMARY TARGET: triage_acuity (ESI 1-5)
# This is what we're predicting in this stage

target_acuity = train_full_df['triage_acuity']  # ESI 1-5 scale

# Features: All columns except patient identifiers and outcome variables
# (disposition and ed_los_hours are downstream outcomes, not used for acuity prediction)
features_df = train_full_df.drop(
    columns=['patient_id', 'triage_acuity', 'disposition', 'ed_los_hours']
)

print(f"\n✓ TARGET FOR ACUITY MODEL: {target_acuity.shape[0]} samples")
print(f"  Acuity classes: {sorted(target_acuity.unique())}")
print(f"  Distribution:")
print(target_acuity.value_counts().sort_index())

print(f"\n✓ FEATURES FOR MODEL: {features_df.shape}")
print(f"  Numerical & categorical variables: {len(features_df.columns)} total")
print(f"  Column names (first 15):")
print(features_df.columns.tolist()[:15])


# SECTION 3: CUSTOM TEXT FEATURE SELECTOR
## Chi-Square + TF-IDF for Clinical Text Analysis

This custom transformer automates medical text feature extraction:

**1. TF-IDF Vectorization**
   - Converts raw chief complaint text into numerical vectors
   - Uses Snowball Stemmer for medical term normalization (e.g., "respiratory" → "respir")
   - Extracts unigrams and bigrams (1-3 word phrases)
   - Removes common English stopwords and rare terms

**2. Chi-Square Feature Selection (One-vs-Rest)**
   - Identifies discriminative keywords per acuity class
   - Example: "respiratory distress", "hypoxia" → high acuity (ESI 1-2)
   - Example: "routine visit", "follow-up" → low acuity (ESI 4-5)
   - Automatically selects top 50 features per class

**3. Clinical Interpretability**
   - Each selected term is labeled with associated acuity classes
   - Output features are named like: `tfidf_respiratory_c1_c2` (predicts ESI 1-2)
   - Improves model explainability for clinical validation


In [ ]:
class Chi2TextFeatureSelector(BaseEstimator, TransformerMixin):
    """
    Extracts and selects the most discriminative clinical keywords from medical text
    using TF-IDF vectorization + Chi-Square statistical feature selection.
    
    This is particularly effective for acuity triage where specific clinical terms
    are strong predictors of severity:
      Example: "respiratory distress", "shock", "altered mental status" → high acuity
      Example: "routine follow-up", "wellness check", "control visit" → low acuity
    
    The Chi-Square test identifies which TF-IDF features are most associated with
    each acuity level using a One-vs-Rest strategy.
    
    Parameters
    ----------
    text_col : str
        Name of the text column to process (default: 'chief_complaint_raw')
    k_per_label : int
        Number of top keywords to select for each acuity class (default: 50)
    min_df : int
        Minimum document frequency - ignores words in <min_df documents (default: 3)
    ngram_range : tuple
        N-gram range: (1,2) = unigrams + bigrams; (1,3) = unigrams + bigrams + trigrams
    
    Attributes
    ----------
    tfidf_vectorizer_ : TfidfVectorizer
        Fitted TF-IDF vectorizer (learned vocabulary)
    selected_indices_ : list
        Column indices in TF-IDF vocabulary selected by Chi-Square test
    feature_names_ : list
        Human-readable names for selected features
        Format: 'tfidf_word_c1_c2' (word predicts classes 1 and 2)
    """
    
    def __init__(self, 
                 text_col: str = 'chief_complaint_raw', 
                 k_per_label: int = 50, 
                 min_df: int = 3,  
                 ngram_range: tuple = (1, 3)):
        self.text_col = text_col
        self.k_per_label = k_per_label
        self.min_df = min_df
        self.ngram_range = ngram_range
        
        # Will be set during fit()
        self.tfidf_vectorizer_ = None
        self.selected_indices_ = None 
        self.feature_names_ = None
        
    def fit(self, X: pd.DataFrame, y) -> 'Chi2TextFeatureSelector':
        """
        Learn the TF-IDF vocabulary and identify top discriminative features per class.
        
        IMPORTANT: Requires target variable 'y' to compute Chi-Square statistics!
        When used in sklearn.compose.ColumnTransformer, call fit_transform(X, y).
        
        Parameters
        ----------
        X : pd.DataFrame
            Feature matrix containing the text column
        y : array-like
            Target variable (acuity labels 1-5 or 0-4)
        """
        if y is None:
            raise ValueError(
                "Target variable 'y' REQUIRED for Chi-Square feature selection!\n"
                "When using ColumnTransformer, call: preprocessor.fit_transform(X_train, y_train)"
            )
        # Ensure X is DataFrame
        if isinstance(X, np.ndarray):
            if X.ndim == 1:
                X = X.reshape(-1, 1)
            X = pd.DataFrame(X, columns=[self.text_col])
        
        # Convert y to numpy array
        y_arr = y.values if isinstance(y, pd.Series) else np.array(y)
        
        # Extract text column and fill NaN with empty string
        text_data = X[self.text_col].fillna('').astype(str)
        
        # ====================================================================
        # STEP 1: Fit TF-IDF Vectorizer with Snowball Stemming
        # ====================================================================
        print(f"📝 Fitting TF-IDF vectorizer with Snowball Stemmer...")
        print(f"   - min_df={self.min_df}, "
              f"ngrams={self.ngram_range}, "
              f"stopwords=English, "
              f"stemmer=Snowball")
        
        # Snowball Stemmer for medical term normalization
        from nltk.stem.snowball import SnowballStemmer
        stemmer = SnowballStemmer('english')
        
        # Custom tokenizer: split → lowercase → stem
        def tokenize_and_stem(text):
            tokens = text.lower().split()
            return [stemmer.stem(token) for token in tokens]
        
        self.tfidf_vectorizer_ = TfidfVectorizer(
            input='content',
            encoding='utf-8',
            lowercase=True,
            stop_words='english',  # Remove common English stopwords
            min_df=self.min_df,  # Ignore rare terms
            ngram_range=self.ngram_range,  # Extract unigrams and bigrams
            tokenizer=tokenize_and_stem  # Apply Snowball stemming to each token
        )
        
        # Compute TF-IDF scores: shape = (n_samples, n_features_tfidf)
        X_tfidf = self.tfidf_vectorizer_.fit_transform(text_data)
        print(f"   ✓ TF-IDF vocabulary size: {X_tfidf.shape[1]} terms")
        
        # ====================================================================
        # STEP 2: Chi-Square Feature Selection (One-vs-Rest)
        # ====================================================================
        # For each class, compute Chi2 scores between TF-IDF features and
        # a binary target (1 = class, 0 = not this class). Select top k features per class.
        
        print(f"📊 Selecting top {self.k_per_label} features per class via Chi-Square test...")
        unique_classes = np.unique(y_arr)
        print(f"   Classes: {unique_classes}")
        
        feature_to_labels = {}  # Track which classes each feature helps predict
        self.feature_to_chi2_scores_ = {}  # Track Chi2 scores per class per feature
        
        for label in unique_classes:
            # Create binary target: 1 if sample belongs to this class, 0 otherwise
            y_binary = (y_arr == label).astype(int)
            
            # Compute Chi2 scores between each TF-IDF feature and binary target
            # chi2() returns (scores, p_values) — we only use scores
            chi2_scores, _ = chi2(X_tfidf, y_binary)
            
            # How many features can we select? (Don't exceed vocabulary size)
            n_features = X_tfidf.shape[1]
            k_safe = min(self.k_per_label, n_features)
            
            if k_safe > 0:
                # Get indices of top k features (highest Chi2 scores)
                top_k_indices = np.argsort(chi2_scores)[-k_safe:]
                
                # Record that these feature indices are predictive for this class
                for idx in top_k_indices:
                    if idx not in feature_to_labels:
                        feature_to_labels[idx] = {}
                        self.feature_to_chi2_scores_[idx] = {}
                    feature_to_labels[idx][label] = chi2_scores[idx]
                    self.feature_to_chi2_scores_[idx][label] = chi2_scores[idx]
        
        # ====================================================================
        # STEP 3: Finalize Selected Indices and Create Feature Names
        # ====================================================================
        # Combine all indices selected across all classes (union)
        self.selected_indices_ = sorted(list(feature_to_labels.keys()))
        
        if not self.selected_indices_:
            print("⚠️  Warning: No features were selected!")
            self.feature_names_ = []
            return self
        
        # Get the actual words/n-grams from the TF-IDF vocabulary
        raw_feature_names = self.tfidf_vectorizer_.get_feature_names_out()
        self.feature_names_ = []
        
        for idx in self.selected_indices_:
            # Get the word/n-gram
            word = raw_feature_names[idx]
            
            # Get Chi2 scores for this feature across all classes
            chi2_per_class = feature_to_labels[idx]
            
            # Find the dominant class (highest Chi2 score)
            dominant_class = max(chi2_per_class, key=chi2_per_class.get)
            
            # Get all classes that predicted this feature, sorted by Chi2 score (descending)
            classes_sorted = sorted(chi2_per_class.keys(), 
                                   key=lambda x: chi2_per_class[x], 
                                   reverse=True)
            
            # Create feature name: dominant class first, then other classes
            # Format: tfidf_word_c0_2_3 where c0 is the dominant class
            classes_suffix = "_".join([f"c{lbl}" for lbl in classes_sorted])
            
            # Final feature name: e.g., "tfidf_apnea_c1_2"
            feature_name = f"tfidf_{word}_{classes_suffix}"
            self.feature_names_.append(feature_name)
        
        print(f"   ✓ Total unique features selected (union): {len(self.selected_indices_)}")
        return self
    
    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """
        Apply learned TF-IDF transformation and filter to selected features.
        """
        if self.tfidf_vectorizer_ is None:
            raise RuntimeError(
                "This transformer has not been fitted yet!\n"
                "Call fit() or fit_transform() first."
            )
        
        # Ensure X is DataFrame
        if isinstance(X, np.ndarray):
            if X.ndim == 1:
                X = X.reshape(-1, 1)
            X = pd.DataFrame(X, columns=[self.text_col])
        
        # Copy to avoid modifying original
        df = X.copy()
        text_data = df[self.text_col].fillna('').astype(str)
        
        # Remove raw text column (we don't want it in the final feature set)
        dfs_to_concat = [df.drop(columns=[self.text_col], errors='ignore')]
        
        if len(self.selected_indices_) > 0:
            # Transform using the full TF-IDF vocabulary learned during fit()
            X_tfidf_full = self.tfidf_vectorizer_.transform(text_data)
            
            # Select only the columns (features) that were chosen during fit()
            X_tfidf_selected = X_tfidf_full[:, self.selected_indices_]
            
            # Convert sparse matrix to dense DataFrame
            df_tfidf = pd.DataFrame(
                X_tfidf_selected.toarray(),  # Convert sparse to dense
                columns=self.feature_names_,  # Use descriptive feature names
                index=df.index  # Preserve original index for alignment
            )
            
            # Add TF-IDF features to concatenation list
            dfs_to_concat.append(df_tfidf)
        
        # Horizontally concatenate: other features + TF-IDF features
        final_df = pd.concat(dfs_to_concat, axis=1)
        return final_df

In [ ]:
# ====================================================================
# ClinicalBERT Embedder for Medical Text Analysis
# ====================================================================
class ClinicalBERTEmbedder(BaseEstimator, TransformerMixin):
    """
    Extracts semantic embeddings from clinical text using ClinicalBERT.
    
    ClinicalBERT is trained on MIMIC-III clinical notes (87K+ records).
    It understands medical terminology, abbreviations, and clinical patterns
    better than generic BERT models.
    
    Uses XPU (Intel GPU) if available for faster inference, falls back to CPU.
    
    Parameters
    ----------
    text_col : str
        Name of the text column to process (default: 'chief_complaint_raw')
    max_length : int
        Maximum token sequence length (default: 128)
    batch_size : int
        Number of samples to process at once (default: 16)
    """
    
    def __init__(self,
                 text_col: str = 'chief_complaint_raw',
                 max_length: int = 128,
                 batch_size: int = 16):
        if not TRANSFORMERS_AVAILABLE:
            raise RuntimeError(
                "HuggingFace transformers library is required.\n"
                "Install with: pip install torch transformers"
            )
        
        self.text_col = text_col
        self.max_length = max_length
        self.batch_size = batch_size
        self.model_ = None
        self.tokenizer_ = None
        self.device = DEVICE  # Use global DEVICE (XPU if available, else CPU)
    
    def fit(self, X, y=None):
        """Load ClinicalBERT model and tokenizer"""
        print(f"🏥 Loading ClinicalBERT (medical-domain embeddings)...")
        print(f"   Device: {self.device} (XPU optimized)" if 'xpu' in str(self.device) else f"   Device: {self.device}")
        
        try:
            # Load ClinicalBERT from HuggingFace
            self.tokenizer_ = AutoTokenizer.from_pretrained('medicalai/ClinicalBERT')
            self.model_ = AutoModel.from_pretrained('medicalai/ClinicalBERT').to(self.device)
            self.model_.eval()
            print(f"   ✓ ClinicalBERT loaded (768-dim embeddings, MIMIC-III trained)")
        except Exception as e:
            print(f"   ⚠️  ClinicalBERT not available: {str(e)[:50]}...")
            print(f"   Falling back to generic BERT...")
            self.tokenizer_ = AutoTokenizer.from_pretrained('distilbert-base-uncased')
            self.model_ = AutoModel.from_pretrained('distilbert-base-uncased').to(self.device)
            self.model_.eval()
            print(f"   ✓ DistilBERT loaded (generic, fast, device={self.device})")
        
        return self
    
    def transform(self, X):
        """Extract [CLS] token embeddings using XPU if available"""
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X, columns=[self.text_col] if X.shape[1] == 1 else None)
        
        texts = X[self.text_col].fillna('').astype(str).tolist()
        embeddings = []
        
        print(f"🏥 Extracting embeddings from {len(texts)} texts (device: {self.device})...")
        
        self.model_.eval()
        with torch.no_grad():
            for i in range(0, len(texts), self.batch_size):
                batch_texts = texts[i:i + self.batch_size]
                
                encoded = self.tokenizer_(
                    batch_texts,
                    max_length=self.max_length,
                    padding='max_length',
                    truncation=True,
                    return_tensors='pt'
                )
                
                input_ids = encoded['input_ids'].to(self.device)
                attention_mask = encoded['attention_mask'].to(self.device)
                
                outputs = self.model_(input_ids, attention_mask=attention_mask)
                cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
                embeddings.append(cls_embeddings)
                
                if (i // self.batch_size + 1) % max(1, len(texts) // (self.batch_size * 5)) == 0:
                    print(f"   Progress: {i + len(batch_texts)}/{len(texts)}")
        
        embeddings_array = np.vstack(embeddings)
        print(f"   ✓ Embeddings shape: {embeddings_array.shape} (device inference complete)")
        
        return embeddings_array


print("✓ ClinicalBERTEmbedder class defined successfully (XPU-enabled)!")

# SECTION 4: ADVANCED FEATURE ENGINEERING

## Feature Engineering Strategy

We create derived features to capture complex relationships in medical data:

### A. Blood Pressure-Derived Features
- **Pulse Pressure Ratio**: `(systolic - diastolic) / mean_arterial_pressure`
  - High ratio → systemic hypertension or vascular stiffness (acuity indicator)
- **MAP to Systolic Ratio**: `mean_arterial_pressure / systolic_bp`
  - Extreme values may indicate shock or malperfusion

### B. Temporal Cyclic Encoding
- **Arrival Hour** and **Arrival Month** are cyclic (hour 23 → hour 0)
- Convert to sin/cos components to preserve circularity
- Sine encoding: `sin(2π * feature / max_value)`
- Example: hour 18 should be close to hours 17 and 19, NOT to hour 18

### C. Comorbidity Interactions
- **Comorbidity Count × Vital Signs**: Higher disease burden + abnormal vitals = higher acuity
- Example: A patient with 5 comorbidities AND tachycardia (hr>100) is higher risk than either alone

### D. Text-Based Risk Indicators
- Words like "apnea", "shock", "hemorrhage" → high acuity
- Words like "followup", "routine", "advice" → low acuity
- Chi2 selection identifies these automatically per acuity level

In [ ]:
# ============================================================================
# Function: Advanced Feature Engineering
# ============================================================================

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create derived features to capture complex medical relationships.
    
    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe with raw features
    
    Returns
    -------
    pd.DataFrame
        Dataframe with new engineered features added
    """
    df = df.copy()  # Avoid modifying original
    
    print("🔧 ENGINEERING FEATURES...")
    
    # ====================================================================
    # A. BLOOD PRESSURE-DERIVED FEATURES
    # ====================================================================
    # These capture vascular and hemodynamic status not obvious from raw vitals
    
    if 'pulse_pressure' in df.columns and 'mean_arterial_pressure' in df.columns:
        # Pulse pressure ratio (avoid division by zero)
        df['pulse_pressure_ratio'] = df['pulse_pressure'] / (df['mean_arterial_pressure'] + 0.1)
        print("   ✓ pulse_pressure_ratio = pulse_pressure / (MAP + 0.1)")
    
    if 'mean_arterial_pressure' in df.columns and 'systolic_bp' in df.columns:
        # MAP to systolic ratio (normal ~0.33-0.40)
        df['map_systolic_ratio'] = df['mean_arterial_pressure'] / (df['systolic_bp'] + 0.1)
        print("   ✓ map_systolic_ratio = MAP / (systolic + 0.1)")
    
    # ====================================================================
    # B. CYCLIC ENCODING FOR TEMPORAL FEATURES
    # ====================================================================
    # Hours and months are cyclic: 23:00 is close to 00:00, not far away
    # Use sin/cos transformation to preserve this circularity
    
    if 'arrival_hour' in df.columns:
        df['arrival_hour_sin'] = np.sin(2 * np.pi * df['arrival_hour'] / 24)
        df['arrival_hour_cos'] = np.cos(2 * np.pi * df['arrival_hour'] / 24)
        print("   ✓ arrival_hour_sin, arrival_hour_cos (cyclic encoding)")
    
    if 'arrival_month' in df.columns:
        df['arrival_month_sin'] = np.sin(2 * np.pi * df['arrival_month'] / 12)
        df['arrival_month_cos'] = np.cos(2 * np.pi * df['arrival_month'] / 12)
        print("   ✓ arrival_month_sin, arrival_month_cos (cyclic encoding)")
    
    # ====================================================================
    # C. COMORBIDITY × VITAL SIGN INTERACTIONS
    # ====================================================================
    # High comorbidity count alone might not be alarming.
    # But high comorbidity × abnormal vitals is a strong risk indicator.
    
    if 'num_comorbidities' in df.columns and 'heart_rate' in df.columns:
        # Flag: high comorbidity + tachycardia (HR > 100)
        df['high_comorbidity_tachycardia'] = (
            (df['num_comorbidities'] > df['num_comorbidities'].quantile(0.75)) & 
            (df['heart_rate'] > 100)
        ).astype(int)
        print("   ✓ high_comorbidity_tachycardia (interaction term)")
    
    if 'num_comorbidities' in df.columns and 'respiratory_rate' in df.columns:
        # Flag: high comorbidity + tachypnea (RR > 20)
        df['high_comorbidity_tachypnea'] = (
            (df['num_comorbidities'] > df['num_comorbidities'].quantile(0.75)) & 
            (df['respiratory_rate'] > 20)
        ).astype(int)
        print("   ✓ high_comorbidity_tachypnea (interaction term)")
    
    # ====================================================================
    # D. VITAL SIGN ABNORMALITY FLAGS
    # ====================================================================
    # Simple categorical indicators for extreme/abnormal values
    
    vital_flags = {
        'heart_rate': (50, 120),           # Normal: 50-120 bpm
        'respiratory_rate': (12, 20),      # Normal: 12-20 breaths/min
        'spo2': (94, 100),                 # Normal: >94%
    }
    
    for vital, (lower, upper) in vital_flags.items():
        if vital in df.columns:
            df[f'{vital}_abnormal'] = (
                (df[vital] < lower) | (df[vital] > upper)
            ).astype(int)
            print(f"   ✓ {vital}_abnormal flag")
    
    # ====================================================================
    # E. NEWS2 SCORE BINNING
    # ====================================================================
    # NEWS2 is already a composite score; binning it creates categorical risk levels
    
    if 'news2_score' in df.columns:
        df['news2_risk_level'] = pd.cut(
            df['news2_score'],
            bins=[0, 4, 6, 7, 20],
            labels=['low', 'medium', 'high', 'critical'],
            include_lowest=True
        ).cat.codes  # Convert to numeric (0, 1, 2, 3)
        print("   ✓ news2_risk_level (binned score)")
    
    # ====================================================================
    # F. AGE-BASED RISK GROUPING
    # ====================================================================
    # Very young (<5 yrs) and very old (>75 yrs) have higher acuity
    
    if 'age' in df.columns:
        df['is_pediatric'] = (df['age'] < 5).astype(int)
        df['is_elderly'] = (df['age'] > 75).astype(int)
        df['is_very_elderly'] = (df['age'] > 85).astype(int)
        print("   ✓ is_pediatric, is_elderly, is_very_elderly flags")
    
    # ====================================================================
    # G. GCS ABNORMALITY (Glasgow Coma Scale)
    # ====================================================================
    # GCS < 14 indicates altered mental status (high acuity)
    
    if 'gcs_total' in df.columns:
        df['gcs_altered'] = (df['gcs_total'] < 14).astype(int)
        print("   ✓ gcs_altered flag (GCS < 14)")
    
    print("✓ FEATURE ENGINEERING COMPLETE!")
    print(f"  New columns added: {len(df.columns) - len(features_df.columns)}")
    
    return df

# Apply feature engineering to training features
features_engineered_df = engineer_features(features_df.copy())

print(f"\n📊 FEATURE MATRIX AFTER ENGINEERING:")
print(f"   Shape: {features_engineered_df.shape}")
print(f"   Columns: {len(features_engineered_df.columns)}")

# SECTION 5: PREPROCESSING & FEATURE ENGINEERING PIPELINE

## Scikit-learn ColumnTransformer Strategy

We split features into 3 types and apply targeted transformations:

| Feature Type | Features | Transformations |
|---|---|---|
| **Categorical** | arrival_mode, sex, insurance_type, age_group, etc. | Impute (most_frequent) → OneHotEncoder |
| **Numerical** | heart_rate, systolic_bp, age, ratios, flags | Impute (constant=-1) → StandardScaler |
| **Text** | chief_complaint_raw | Impute → ClinicalBERT + Chi2 TF-IDF |

## Feature Engineering Steps

**A. Blood Pressure Features**: Pulse pressure ratio, MAP-to-systolic ratio
**B. Cyclic Encoding**: Arrival hour/month as sin/cos (circularity preservation)
**C. Comorbidity Interactions**: High comorbidity count × abnormal vitals
**D. Vital Sign Flags**: Binary abnormality indicators (HR, RR, SpO2)
**E. NEWS2 Risk Binning**: Composite score → low/medium/high/critical
**F. Age-Based Flags**: Pediatric, elderly, very elderly patients
**G. GCS Abnormality**: Glasgow Coma Scale < 14 (altered mental status)

**Output**: ~200-250 features after preprocessing (categorical one-hot + numerical + BERT embeddings + Chi2 keywords)


In [ ]:
# ============================================================================
# Identify feature types in engineered feature matrix
# ============================================================================
text_feature = 'chief_complaint_raw'

# Define KNOWN categorical features from original dataset
known_categorical = ['site_id', 'triage_nurse_id', 'arrival_mode', 'arrival_day', 
                     'arrival_season', 'shift', 'age_group', 
                      'sex', 'language', 'insurance_type', 'transport_origin',
                      'pain_location', 'pain_severity', 'pain_onset',
                      'mental_status_triage', 'acuity_reason_1', 'acuity_reason_2', 
                      'chief_complaint_raw', 'medication_allergy', 'ed_visit_reason']

# Categorical: object (string) type OR in known categorical list, excluding text
categorical_features = [
    col for col in features_engineered_df.columns 
    if (features_engineered_df[col].dtype == 'object' or col in known_categorical)
    and col != text_feature
]

# Ensure ALL object dtype columns are in categorical (safety check)
all_object_cols = features_engineered_df.select_dtypes(include=['object']).columns.tolist()
categorical_features = list(set(categorical_features + [c for c in all_object_cols if c != text_feature]))

# Numerical: everything except categorical and text
numerical_features = [
    col for col in features_engineered_df.columns
    if col not in categorical_features and col != text_feature
]

print(f"\n🔍 FEATURE TYPE CLASSIFICATION:")
print(f"   Categorical ({len(categorical_features)}): {categorical_features[:5]}...")
print(f"   Numerical ({len(numerical_features)}):   {numerical_features[:5]}...")
print(f"   Text (1):                    {text_feature}")

# ============================================================================
# Train-Validation Split (80-20, stratified by acuity)
# ============================================================================
# Stratification ensures that both train and validation have similar acuity distributions
# This is CRITICAL for imbalanced datasets like triage acuity

X_train_raw, X_val_raw, y_train_acuity, y_val_acuity = train_test_split(
    features_engineered_df,
    target_acuity,
    train_size=0.8,
    random_state=42,
    stratify=target_acuity  # Maintain acuity distribution in both sets
)

print(f"\n📊 TRAIN-VALIDATION SPLIT:")
print(f"   Training set:   {X_train_raw.shape[0]} samples ({X_train_raw.shape[0]/len(features_engineered_df)*100:.1f}%)")
print(f"   Validation set: {X_val_raw.shape[0]} samples ({X_val_raw.shape[0]/len(features_engineered_df)*100:.1f}%)")

print(f"\n✓ ACUITY DISTRIBUTION (train set):")
print(y_train_acuity.value_counts().sort_index())
print(f"\n✓ ACUITY DISTRIBUTION (val set):")
print(y_val_acuity.value_counts().sort_index())

# ============================================================================
# Build ColumnTransformer Preprocessing Pipeline
# ============================================================================
print(f"\n🔧 BUILDING PREPROCESSING PIPELINE...")

# Categorical processor: impute mode → one-hot encoding
categorical_pipeline = Pipeline([
    ('imputer',  SimpleImputer(strategy='most_frequent')),
    ('onehot',   OneHotEncoder(handle_unknown='ignore', sparse_output=True))
])

# Numerical processor: impute constant (-1) → standardize
numerical_pipeline = Pipeline([
    ('imputer',  SimpleImputer(strategy='constant', fill_value=-1)),
    ('scaler',   StandardScaler())
])

# Text processor: ClinicalBERT + Chi2 medical keywords
# Combines medical-domain embeddings + discriminative text features
from sklearn.pipeline import FeatureUnion

text_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('features', FeatureUnion([
        ('clinical_bert', ClinicalBERTEmbedder(
            text_col=text_feature,
            max_length=128,
            batch_size=16
        )),
        ('chi2_tfidf', Chi2TextFeatureSelector(
            text_col=text_feature,
            k_per_label=50,
            min_df=3,
            ngram_range=(1, 3)
        ))
    ]))
])

print("✓ Text pipeline (ClinicalBERT + Chi2) configured!")

# Combine all processors
preprocessor = ColumnTransformer(
    transformers=[
        ('cat',  categorical_pipeline, categorical_features),
        ('num',  numerical_pipeline,   numerical_features),
        ('text', text_pipeline,        [text_feature])
    ],
    remainder='drop'  # Drop any columns not listed above
)

print("✓ Pipeline components defined!")

# ============================================================================
# FIT THE PREPROCESSOR ON TRAINING DATA (WITH LABELS!)
# ============================================================================
# ⚠️ CRITICAL: We pass y_train_acuity to fit_transform() so Chi2 selector gets labels

print(f"\n🚀 FITTING PREPROCESSOR ON TRAINING DATA...")
X_train_preprocessed = preprocessor.fit_transform(X_train_raw, y_train_acuity)

# Transform validation data (WITHOUT retraining text selector)
X_val_preprocessed = preprocessor.transform(X_val_raw)

print(f"✓ Preprocessing complete!")
print(f"   Training feature matrix:   {X_train_preprocessed.shape}")
print(f"   Validation feature matrix: {X_val_preprocessed.shape}")

# SECTION 6: ACUITY PREDICTION MODEL - LightGBM Classifier

## Why LightGBM for Acuity Prediction?

**LightGBM** (Light Gradient Boosting Machine) is a powerful gradient boosting framework optimized for healthcare classification:

**Advantages**:
1. **Fast Training**: Leaf-wise tree growth strategy (vs. level-wise) → fewer trees needed
2. **Memory Efficient**: Handles large datasets with minimal RAM overhead
3. **Native Categorical Support**: Built-in categorical encoding (no separate one-hot needed)
4. **Strong Performance**: Often outperforms Random Forest on real-world medical datasets
5. **Interpretability**: Feature importance rankings and SHAP value compatibility

**Model Configuration**:
- **Objective**: `multiclass` softmax for multi-class ordinal classification (ESI 1-5)
- **Learning Rate**: 0.1 (conservative to prevent overfitting on medical data)
- **Num Leaves**: 31 (balanced tree complexity)
- **Regularization**: L1=0.1, L2=0.1 to reduce overfitting

## Evaluation Metrics for Acuity Prediction

| Metric | Interpretation |
|--------|---|
| **Accuracy** | % of correct acuity predictions (baseline) |
| **Cohen's Kappa (Linear)** | Agreement corrected for chance (ordinal distance weighted) |
| **Cohen's Kappa (Quadratic)** | Penalizes large errors more (ESI 1→5 worse than 1→2) |
| **Undertriage %** | Predictions WORSE than actual (👎 dangerous, unsafe) |
| **Overtriage %** | Predictions BETTER than actual (👍 resource-intensive but safer) |
| **Correct %** | Predictions EQUAL to actual (✓ perfect) |

**Clinical Safety**: Undertriage is unacceptable in ED triage. Better to predict high acuity when uncertain.


In [ ]:
# ============================================================================
# TRAIN LIGHTGBM CLASSIFIER FOR ACUITY PREDICTION
# ============================================================================
# Single-stage model: Features (vitals, demographics, text) → Acuity (ESI 1-5)

print("\n" + "="*80)
print("ACUITY PREDICTION MODEL: TRAINING PHASE")
print("="*80)

# Convert sparse matrix to dense for training
X_train_dense = X_train_preprocessed.toarray() if hasattr(X_train_preprocessed, 'toarray') else X_train_preprocessed
X_val_dense = X_val_preprocessed.toarray() if hasattr(X_val_preprocessed, 'toarray') else X_val_preprocessed

# Convert Series to numpy arrays for LightGBM
y_train_values = y_train_acuity.values if hasattr(y_train_acuity, 'values') else y_train_acuity
y_val_values = y_val_acuity.values if hasattr(y_val_acuity, 'values') else y_val_acuity

# Initialize LightGBM for multi-class acuity classification (ESI 1-5)
lgbm_acuity = LGBMClassifier(
    objective='multiclass',
    num_class=5,                       # ESI 1-5 = 5 classes (or 0-4 in code)
    num_leaves=31,                     # Balanced tree complexity
    learning_rate=0.1,                 # Conservative learning (prevent overfitting)
    n_estimators=200,                  # Number of boosting stages
    max_depth=7,                       # Maximum tree depth
    min_child_samples=5,               # Minimum samples per leaf (prevent overfit)
    lambda_l1=0.1,                     # L1 regularization
    lambda_l2=0.1,                     # L2 regularization
    feature_fraction=0.8,              # Use 80% of features per tree (bagging)
    bagging_fraction=0.8,              # Use 80% of samples per tree
    bagging_freq=5,                    # Bagging frequency (every 5 trees)
    verbose=-1,                        # Suppress training logs
    random_state=42                    # Reproducibility
)

print(f"\n🤖 TRAINING LIGHTGBM ACUITY MODEL")
print(f"   Training samples: {X_train_dense.shape[0]:,}")
print(f"   Features: {X_train_dense.shape[1]:,}")
print(f"   Acuity classes: {np.unique(y_train_values)} (0=ESI1, 1=ESI2, ..., 4=ESI5)")

lgbm_acuity.fit(
    X_train_dense, 
    y_train_values,
    eval_set=[(X_val_dense, y_val_values)],
    eval_metric='multi_logloss'
)
print("✓ Model training complete!")

# ============================================================================
# EVALUATE ACUITY MODEL ON VALIDATION SET
# ============================================================================

y_pred_acuity_val = lgbm_acuity.predict(X_val_dense)
y_prob_acuity_val = lgbm_acuity.predict_proba(X_val_dense)

# Compute evaluation metrics
acc = accuracy_score(y_val_values, y_pred_acuity_val)
kappa_linear = cohen_kappa_score(y_val_values, y_pred_acuity_val, weights='linear')
kappa_quadratic = cohen_kappa_score(y_val_values, y_pred_acuity_val, weights='quadratic')

# F1 scores (macro = unweighted, weighted = by class frequency)
f1_macro = f1_score(y_val_values, y_pred_acuity_val, average='macro')
f1_weighted = f1_score(y_val_values, y_pred_acuity_val, average='weighted')

# Clinical Safety Metrics
# - Undertriage: predicted < actual (dangerous! -> ES I-3 predicted as ESI-4)
# - Overtriage: predicted > actual (safe but wasteful -> ESI-3 predicted as ESI-2)
# - Correct: predicted = actual (ideal)
undertriage_rate = np.sum(y_pred_acuity_val < y_val_values) / len(y_val_values)
overtriage_rate = np.sum(y_pred_acuity_val > y_val_values) / len(y_val_values)
correct_rate = np.sum(y_pred_acuity_val == y_val_values) / len(y_val_values)

print(f"\n📊 ACUITY MODEL PERFORMANCE (Validation Set):")
print(f"   Accuracy:               {acc:.4f} ({acc*100:.2f}%)")
print(f"   Cohen's Kappa (linear): {kappa_linear:.4f} (ordinal distance weighted)")
print(f"   Cohen's Kappa (quad):   {kappa_quadratic:.4f} (penalizes large errors more)")
print(f"   F1 Score (macro):       {f1_macro:.4f} (unweighted average)")
print(f"   F1 Score (weighted):    {f1_weighted:.4f} (frequency-weighted)")

print(f"\n⚠️  CLINICAL SAFETY METRICS (CRITICAL):")
print(f"   Undertriage rate:  {undertriage_rate*100:.2f}% (predicted < actual) ⚠️ DANGEROUS")
print(f"   Correct rate:      {correct_rate*100:.2f}% (predicted = actual) ✓")
print(f"   Overtriage rate:   {overtriage_rate*100:.2f}% (predicted > actual)   (conservative but safer)")

# Confusion Matrix
print(f"\n📋 CONFUSION MATRIX (Validation Set):")
print(f"   Rows=Actual ESI, Columns=Predicted ESI")
cm = confusion_matrix(y_val_values, y_pred_acuity_val)
print(pd.DataFrame(cm, index=[f"ESI {i}" for i in range(1, 6)], 
                    columns=[f"ESI {i}" for i in range(1, 6)]))

# Classification Report
print(f"\n📈 PER-CLASS PERFORMANCE (Precision, Recall, F1):")
print(classification_report(y_val_values, y_pred_acuity_val, 
                           target_names=[f"ESI {i}" for i in range(1, 6)]))

print("\n✓ Acuity model evaluation complete!")


In [ ]:
# ============================================================================
# EXTRACT AND ORGANIZE FEATURE NAMES FROM PREPROCESSING PIPELINE
# ============================================================================
# Goal: Map each model feature back to its original source for interpretability

print("\n" + "="*80)
print("EXTRACTING FEATURE NAMES FROM PREPROCESSOR")
print("="*80)

feature_names_stage1 = []

# Get all transformers from the preprocessor
transformers = preprocessor.transformers_

for name, transformer, columns in transformers:
    if name == 'cat':
        # OneHotEncoded categorical features: get names from OneHotEncoder
        print(f"\n🏷️  CATEGORICAL FEATURES (one-hot encoded):")
        try:
            onehot = transformer.named_steps['onehot']
            cat_feature_names = onehot.get_feature_names_out(columns)
            feature_names_stage1.extend(cat_feature_names)
            print(f"   ✓ Extracted {len(cat_feature_names)} one-hot encoded features")
            if len(cat_feature_names) > 0:
                print(f"     Examples: {list(cat_feature_names[:3])}")
        except Exception as e:
            print(f"   ⚠️  Could not extract categorical names: {str(e)[:50]}")
            feature_names_stage1.extend([f"cat_{i}" for i in range(len(columns))])
    
    elif name == 'num':
        # Numerical features (standardized but unchanged names)
        print(f"\n📊 NUMERICAL FEATURES (standardized):")
        feature_names_stage1.extend(columns)
        print(f"   ✓ Extracted {len(columns)} numerical feature names")
        print(f"     Examples: {columns[:5]}")
    
    elif name == 'text':
        # Text features: ClinicalBERT embeddings + Chi2 TF-IDF keywords
        print(f"\n📝 TEXT FEATURES (Clinical NLP):")
        
        try:
            text_pipeline = transformer
            feature_union = text_pipeline.named_steps['features']
            transformers_list = feature_union.transformer_list
            
            chi2_names = []
            bert_names = []
            
            for fu_name, fu_transformer in transformers_list:
                if 'chi2' in fu_name.lower():
                    # Extract Chi2 selected keywords
                    if hasattr(fu_transformer, 'feature_names_'):
                        chi2_names = fu_transformer.feature_names_ if fu_transformer.feature_names_ else []
                        print(f"   ✓ Chi2 TF-IDF keywords: {len(chi2_names)} terms selected")
                        if chi2_names:
                            print(f"     Examples: {chi2_names[:3]}")
                    else:
                        print(f"   ⚠️  Chi2 transformer missing feature_names_")
                
                elif 'bert' in fu_name.lower():
                    # ClinicalBERT: 768-dimensional embeddings
                    n_bert = 768
                    bert_names = [f"clinical_bert_{i:03d}" for i in range(n_bert)]
                    print(f"   ✓ ClinicalBERT: {len(bert_names)} embedding dimensions (MIMIC-III trained)")
                    print(f"     Format: clinical_bert_000 to clinical_bert_767")
            
            # Combine: BERT embeddings + Chi2 keywords
            feature_names_stage1.extend(bert_names)
            feature_names_stage1.extend(chi2_names)
            print(f"   ✓ Total text features: {len(bert_names)} + {len(chi2_names)} = {len(bert_names) + len(chi2_names)}")
            
        except Exception as e:
            print(f"   ⚠️  Error extracting text features: {str(e)[:100]}")
            n_text = 768 + 50  # Estimate
            feature_names_stage1.extend([f"text_feature_{i}" for i in range(n_text)])

print(f"\n{'='*80}")
print(f"✓ Total feature names extracted: {len(feature_names_stage1)}")
print(f"   Expected from model: {X_train_dense.shape[1]}")

# Validate feature count matches
if len(feature_names_stage1) != X_train_dense.shape[1]:
    print(f"\n⚠️  MISMATCH DETECTED:")
    print(f"   Named features: {len(feature_names_stage1)}")
    print(f"   Model features: {X_train_dense.shape[1]}")
    
    if len(feature_names_stage1) < X_train_dense.shape[1]:
        n_missing = X_train_dense.shape[1] - len(feature_names_stage1)
        print(f"   → Padding with {n_missing} generic names")
        feature_names_stage1.extend([f"feature_{i}" for i in range(n_missing)])
    else:
        print(f"   → Trimming to {X_train_dense.shape[1]} features")
        feature_names_stage1 = feature_names_stage1[:X_train_dense.shape[1]]

print(f"\n✅ Final feature names: {len(feature_names_stage1)}")


In [ ]:
# # expand X_train_dense with feature names for interpretability
# X_train_df = pd.DataFrame(X_train_dense, columns=feature_names_stage1)
# # expand X_val_dense with feature names for interpretability
# X_val_df = pd.DataFrame(X_val_dense, columns=feature_names_stage1)
# # Save them to send to Timofei
# X_train_df.to_csv("X_train_preprocessed_tim.csv", index=False)
# X_val_df.to_csv("X_val_preprocessed_tim.csv", index=False)

In [ ]:
# ============================================================================
# FEATURE IMPORTANCE ANALYSIS - LightGBM Built-in Importance
# ============================================================================
# LightGBM scores features by how often they are used + their contribution
# This identifies which clinical variables matter most for acuity prediction

feature_importances = lgbm_acuity.feature_importances_
feature_importance_dict = dict(zip(feature_names_stage1, feature_importances))

# Sort features by importance (descending)
sorted_features = sorted(feature_importance_dict.items(), key=lambda x: x[1], reverse=True)

print(f"\n📊 TOP 30 MOST IMPORTANT FEATURES FOR ACUITY PREDICTION:")
print(f"{'Rank':<5} {'Feature':<50} {'Importance':>10}")
print(f"{'-'*65}")
for i, (feature, importance) in enumerate(sorted_features[:30]):
    print(f"{i+1:4d}. {feature:50s} {importance:10.4f}")


# SECTION 7: MODEL EXPLAINABILITY - SHAP VALUES FOR CLINICAL INTERPRETABILITY

## What are SHAP Values?

**SHAP** (SHapley Additive exPlanations) is a unified approach to model interpretability based on game theory:

**Key Concepts**:
1. **Additive Attribution**: Each feature's contribution to the final prediction is quantified
2. **Shapley Values**: Fair, stable approach from cooperative game theory
3. **Local Explanations**: Explains individual predictions (not just global patterns)
4. **Clinical Trust**: Helps clinicians understand and validate triage decisions

**Why SHAP for Acuity Triage?**
- Transparency: "Why was this patient assigned ESI-3?"
- Validation: Clinicians can verify if model reasoning matches medical knowledge
- Safety: Detect biased or unexpected decision patterns
- Confidence: Trustworthy AI increases clinical adoption

## SHAP Visualization Types

| Visualization | Purpose | Clinical Use |
|---|---|---|
| **Summary Plot** | Feature importance per acuity class | Understand what drives each ESI level |
| **Dependence Plot** | Feature value vs model impact | Discover non-linear vital sign relationships |
| **Force Plot** | Individual patient explanation | "Why was this patient ESI-2?" |
| **Waterfall Plot** | Step-by-step prediction breakdown | Detailed clinical reasoning trace |

## Example: SHAP in Clinical Practice

**Patient Case**: Predicted ESI-2 (High Acuity)
- 🔵 **Positive SHAP** (pushes toward high acuity):
  - Chief complaint: "respiratory distress" (+0.35)
  - Heart rate: 115 bpm (+0.18)
  - Respiratory rate: 24 breaths/min (+0.12)
  - **Net effect**: High acuity diagnosis justified ✓

- 🔴 **Negative SHAP** (pushes toward low acuity):
  - Age: 28 years (-0.05)
  - No comorbidities (-0.02)
  - **Conclusion**: Vitals outweigh demographics → ESI-2 correct

**Result**: Clinician confidence in model decision increases, enabling safe clinical deployment.


In [ ]:
# ============================================================================
# EXPLAINABILITY ANALYSIS: SHAP VALUES FOR MODEL TRANSPARENCY
# ============================================================================
# SHAP values explain INDIVIDUAL predictions:
# "Why was patient X assigned acuity level ESI-3?"
# Shows which features pushed the decision toward each acuity class

print("\n" + "="*80)
print("EXPLAINABILITY: SHAP VALUES FOR ACUITY MODEL")
print("="*80)

print(f"\n📊 Importing SHAP library for model interpretability...")

import shap
print("✓ SHAP imported successfully")

# ====================================================================
# CREATE SHAP EXPLAINER FOR LIGHTGBM ACUITY MODEL
# ====================================================================
print(f"\n🔍 Creating TreeExplainer for LightGBM...")

# Use sample of training data for efficiency (SHAP computation intensive)
sample_size = min(1000, X_train_dense.shape[0])
sample_indices = np.random.choice(X_train_dense.shape[0], sample_size, replace=False)
X_sample = X_train_dense[sample_indices]
y_sample = y_train_values[sample_indices]

# Create SHAP explainer: identifies how each feature contributes to predictions
explainer = shap.TreeExplainer(lgbm_acuity)
shap_values = explainer.shap_values(X_sample)

print(f"✓ SHAP Explainer created successfully")
print(f"   Sample size: {X_sample.shape[0]:,} patients")
print(f"   SHAP values type: {type(shap_values)}")

if isinstance(shap_values, list):
    print(f"   Format: List of {len(shap_values)} arrays (one per class)")
    print(f"   Shape per class: {shap_values[0].shape if len(shap_values) > 0 else 'N/A'}")
else:
    print(f"   Format: Single array")
    print(f"   Shape: {shap_values.shape}")


In [ ]:

# ====================================================================
# SHAP VISUALIZATIONS - Feature Analysis and Explanations
# ====================================================================
print("\n" + "="*80)
print("📊 GENERATING SHAP VISUALIZATIONS")
print("="*80)

# ====================================================================
# 1. Generate Feature Names if not available
# ====================================================================
if 'feature_names_stage1' not in globals():
    print("⚠️  feature_names_stage1 not found, generating generic names...")
    n_features = X_train_dense.shape[1]
    feature_names_stage1 = [f"feature_{i:04d}" for i in range(n_features)]
    
    # Try to add descriptive names for known features
    n_bert = sum(1 for f in feature_names_stage1 if f.startswith('clinical_bert_')) if 'feature_names_stage1' in globals() else 768
    n_chi2 = sum(1 for f in feature_names_stage1 if f.startswith('tfidf_')) if 'feature_names_stage1' in globals() else n_features - 900
    
    feature_names_stage1_corrected = []
    feature_names_stage1_corrected.extend([f"BERT_{i:03d}" for i in range(768)])
    if n_features > 768:
        feature_names_stage1_corrected.extend([f"tfidf_{i:03d}" for i in range(n_features - 768)])
    
    feature_names_stage1 = feature_names_stage1_corrected[:n_features]
    print(f"   Generated {len(feature_names_stage1)} feature names")

print(f"✓ Total features: {len(feature_names_stage1)}")
print(f"✓ Model features: {X_train_dense.shape[1]}")
print(f"✓ SHAP values shape: {shap_values.shape}")

# ====================================================================
# 2. Understand SHAP Format
# ====================================================================
print(f"\n🔍 SHAP FORMAT ANALYSIS:")
print(f"   Type: {type(shap_values)}")
print(f"   Shape: {shap_values.shape}")
print(f"   Is list: {isinstance(shap_values, list)}")

# For multiclass LightGBM with TreeExplainer:
# - If shape is (n_samples, n_features, n_classes): multiclass format
# - If shape is (n_samples, n_features): binary or single output

if len(shap_values.shape) == 3:
    n_samples, n_features, n_classes = shap_values.shape
    print(f"   Format: MULTICLASS (samples={n_samples}, features={n_features}, classes={n_classes})")
    is_multiclass = True
else:
    print(f"   Format: SINGLE OUTPUT or BINARY")
    is_multiclass = False

# ====================================================================
# 3. SHAP FEATURE IMPORTANCE ANALYSIS PER CLASS
# ====================================================================
if is_multiclass:
    print(f"\n{'═'*80}")
    print("📊 SHAP SUMMARY PLOTS - FEATURE IMPORTANCE PER CLASSE")
    print(f"{'═'*80}")
    
    n_classes = shap_values.shape[2]
    class_names = [f"ESI {i+1}" for i in range(n_classes)]
    
    # Create subplots for each class
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    for class_idx in range(min(n_classes, 6)):
        print(f"\n🔍 Class {class_idx} ({class_names[class_idx]}) - Top 15 features:")
        
        # Get SHAP values for this class: shape = (n_samples, n_features)
        class_shap = shap_values[:, :, class_idx]
        
        # Calculate mean absolute SHAP value per feature
        feature_importance = np.abs(class_shap).mean(axis=0)
        
        # Get top 15 features
        top_15_idx = np.argsort(feature_importance)[-15:][::-1]
        
        # Use feature names
        top_15_names = [feature_names_stage1[i] if i < len(feature_names_stage1) else f"Feature {i}" 
                       for i in top_15_idx]
        top_15_importance = feature_importance[top_15_idx]
        
        # Print top features
        for rank, (fname, imp) in enumerate(zip(top_15_names, top_15_importance), 1):
            print(f"   {rank:2d}. {fname:50s} → {imp:.4f}")
        
        # Bar plot on subplot
        axes[class_idx].barh(range(15), top_15_importance, color='steelblue')
        axes[class_idx].set_yticks(range(15))
        axes[class_idx].set_yticklabels(top_15_names, fontsize=8)
        axes[class_idx].set_xlabel('Mean |SHAP value|')
        axes[class_idx].set_title(f'{class_names[class_idx]} - Top 15 Features', fontweight='bold')
        axes[class_idx].invert_yaxis()
        axes[class_idx].grid(axis='x', alpha=0.3)
    
    # Hide empty subplot
    if n_classes < 6:
        axes[n_classes].axis('off')
    
    plt.tight_layout()
    plt.savefig('shap_summary_per_class.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("\n✓ Saved: shap_summary_per_class.png")
else:
    print(f"\n⚠️  Single output format detected (not multiclass), creating simplified analysis...")
    
    # Calculate feature importance from single output
    feature_importance = np.abs(shap_values).mean(axis=0)
    top_20_idx = np.argsort(feature_importance)[-20:][::-1]
    top_20_names = [feature_names_stage1[i] if i < len(feature_names_stage1) else f"Feature {i}" 
                   for i in top_20_idx]
    top_20_importance = feature_importance[top_20_idx]
    
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.barh(range(20), top_20_importance, color='steelblue')
    ax.set_yticks(range(20))
    ax.set_yticklabels(top_20_names, fontsize=9)
    ax.set_xlabel('Mean |SHAP value|')
    ax.set_title('Top 20 Most Important Features (SHAP)', fontweight='bold', fontsize=12)
    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('shap_summary.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("\n✓ Saved: shap_summary.png")


In [ ]:

# ====================================================================
# SHAP FORCE PLOTS - INDIVIDUAL PATIENT EXPLANATIONS
# ====================================================================
# Force plots show how features "push" each prediction toward a specific acuity level
# Green (positive) = features that increase acuity prediction
# Red (negative) = features that decrease acuity prediction

print("\n" + "="*80)
print("🎯 EXPLAINING INDIVIDUAL ACUITY PREDICTIONS")
print("="*80)

# Ensure feature_names_stage1 exists
if 'feature_names_stage1' not in globals():
    n_features = X_train_dense.shape[1]
    feature_names_stage1 = [f"BERT_{i:03d}" if i < 768 else f"tfidf_{i-768:03d}" 
                           for i in range(n_features)]

# Select diverse patients to explain
sample_to_explain = [0, 50, 100, 150, 200]
class_names = [f"ESI {i}" for i in range(1, 6)]

print(f"\n📍 Analyzing predictions for {len(sample_to_explain)} sample patients:")
print(f"   (Each shows which clinical features influenced acuity classification)\n")

# Handle both list and array format for SHAP values
if isinstance(shap_values, list):
    shap_array = np.array([sv for sv in shap_values])
else:
    shap_array = shap_values

# Get base value (model baseline prediction)
base_value = explainer.expected_value
print(f"Base value (model baseline): {base_value if not isinstance(base_value, list) else f'List of {len(base_value)} values'}")

# Explain each sample
for sample_idx in sample_to_explain:
    print(f"\n{'─'*80}")
    print(f"📌 PATIENT #{sample_idx}:")
    
    # Get prediction for this sample
    sample_X = X_sample[sample_idx:sample_idx+1]
    pred = lgbm_acuity.predict(sample_X)[0]
    probas = lgbm_acuity.predict_proba(sample_X)[0]
    
    print(f"   Predicted Acuity: {class_names[pred]} ")
    print(f"   Probability: {', '.join([f'{class_names[i]}={p:.1%}' for i, p in enumerate(probas)])}")
    
    # Get actual acuity 
    actual = y_sample[sample_idx]
    print(f"   Actual Acuity: {class_names[actual]}")
    
    # Extract SHAP values for predicted class
    if isinstance(shap_values, list):
        pred_class_shap = shap_values[pred][sample_idx]
    else:
        # If 3D array: (n_samples, n_features, n_classes)
        if len(shap_array.shape) == 3:
            pred_class_shap = shap_array[sample_idx, :, pred]
        else:
            pred_class_shap = shap_array[sample_idx]
    
    # Get top contributing features
    print(f"\n   🔵 Top features pushing TOWARD acuity (positive SHAP):")
    top_positive_idx = np.argsort(pred_class_shap)[-5:][::-1]
    for rank, feat_idx in enumerate(top_positive_idx, 1):
        feat_val = sample_X[0, feat_idx]
        shap_val = pred_class_shap[feat_idx]
        break
    
    print(f"\n   🔴 Top features pushing AWAY from acuity (negative SHAP):")
    top_negative_idx = np.argsort(pred_class_shap)[:5]
    for rank, feat_idx in enumerate(top_negative_idx, 1):
        feat_val = sample_X[0, feat_idx]
        shap_val = pred_class_shap[feat_idx]
        feat_name = feature_names_stage1[feat_idx] if feat_idx < len(feature_names_stage1) else f"Feature {feat_idx}"
        break

print(f"\n✓ Individual patient explanations complete!")


In [ ]:

# ====================================================================
# SHAP DEPENDENCE PLOTS - FEATURE-PREDICTION RELATIONSHIPS
# ====================================================================
# Shows: Does higher vital value → higher acuity?
# Non-linear relationships may reveal important clinical thresholds
# Example: HR 70-90 neutral, HR >110 strongly pushes toward high acuity

print("\n" + "="*80)
print("📈 SHAP DEPENDENCE PLOTS - FEATURE IMPACT ANALYSIS")
print("="*80)

# Ensure feature_names_stage1 is defined
if 'feature_names_stage1' not in globals():
    n_features = X_train_dense.shape[1]
    feature_names_stage1 = [f"BERT_{i:03d}" if i < 768 else f"tfidf_{i-768:03d}" 
                           for i in range(n_features)]

# Handle SHAP values format
if isinstance(shap_values, list):
    shap_array = np.array([sv for sv in shap_values])
else:
    shap_array = shap_values

# Select top features by global importance
if len(shap_array.shape) == 3:  # Multiclass: (n_samples, n_features, n_classes)
    all_features_importance = np.abs(shap_array).mean(axis=(0, 2))
else:
    all_features_importance = np.abs(shap_array).mean(axis=0)

top_10_features_idx = np.argsort(all_features_importance)[-10:][::-1]

print(f"✓ Top 10 most important features (averaged across all classes):")
for rank, feat_idx in enumerate(top_10_features_idx[:10], 1):
    feat_name = feature_names_stage1[feat_idx] if feat_idx < len(feature_names_stage1) else f"Feature {feat_idx}"
    importance = all_features_importance[feat_idx]
    print(f"   {rank:2d}. {feat_name:50s} ({importance:.4f})")

# Create dependence plots for top 3 features
print(f"\n📊 Creating dependence plots for top 3 features...")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for plot_idx, feat_idx in enumerate(top_10_features_idx[:3]):
    # Get feature values from sample data
    feat_values = X_sample[:, feat_idx]
    
    # Get SHAP values for each sample's predicted class
    shap_vals_for_feat = []
    for sample_i in range(len(X_sample)):
        pred_class = int(lgbm_acuity.predict(X_sample[sample_i:sample_i+1])[0])
        pred_class = np.clip(pred_class, 0, 4)  # Ensure valid range
        
        if isinstance(shap_values, list):
            sv = shap_values[pred_class][sample_i, feat_idx]
        elif len(shap_array.shape) == 3:
            sv = shap_array[sample_i, feat_idx, pred_class]
        else:
            sv = shap_array[sample_i, feat_idx]
        shap_vals_for_feat.append(sv)
    
    shap_vals_for_feat = np.array(shap_vals_for_feat)
    
    # Scatter plot: Feature value vs SHAP value
    scatter = axes[plot_idx].scatter(feat_values, shap_vals_for_feat, 
                                     c=shap_vals_for_feat, cmap='RdBu_r', 
                                     alpha=0.6, s=50, edgecolor='k', linewidth=0.5)
    axes[plot_idx].axhline(y=0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    
    feat_name = feature_names_stage1[feat_idx] if feat_idx < len(feature_names_stage1) else f"Feature {feat_idx}"
    axes[plot_idx].set_xlabel(f'{feat_name} (feature value)', fontsize=10)
    axes[plot_idx].set_ylabel('SHAP value (acuity impact)', fontsize=10)
    axes[plot_idx].set_title(f'Dependence: {feat_name}', fontweight='bold', fontsize=11)
    axes[plot_idx].grid(alpha=0.3)
    
    # Add colorbar
    cbar = plt.colorbar(scatter, ax=axes[plot_idx])
    cbar.set_label('SHAP value', fontsize=9)

plt.tight_layout()
plt.savefig('shap_dependence_plots.png', dpi=100, bbox_inches='tight')
plt.show()
print("\n✓ Saved: shap_dependence_plots.png")


In [ ]:

# ====================================================================
# COMPREHENSIVE ACUITY MODEL ANALYSIS DASHBOARD
# ====================================================================
# Unified visualization showing:
# 1. SHAP value distributions per acuity class
# 2. Per-class performance metrics (Precision, Recall, F1)
# 3. Feature type distribution (embeddings, keywords, clinical vars)
# 4. SHAP value ranges and patterns
# 5. Confusion matrix for prediction accuracy
# 6. Top global feature importance
# 7. Model statistics and clinical safety metrics

print("\n" + "="*80)
print("📊 COMPREHENSIVE ACUITY MODEL ANALYSIS DASHBOARD")
print("="*80)

# Ensure feature_names_stage1 is defined
if 'feature_names_stage1' not in globals():
    n_features = X_train_dense.shape[1]
    feature_names_stage1 = [f"BERT_{i:03d}" if i < 768 else f"tfidf_{i-768:03d}" 
                           for i in range(n_features)]

class_names = [f"ESI {i}" for i in range(1, 6)]

# Handle SHAP values format
if isinstance(shap_values, list):
    shap_array = np.array([sv for sv in shap_values])
else:
    shap_array = shap_values

# Create comprehensive dashboard
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.35)

# 1. SHAP Distribution by Class (top-left)
ax1 = fig.add_subplot(gs[0, 0])
class_shap_means = []
if len(shap_array.shape) == 3:
    for class_idx in range(5):
        mean_shap = np.abs(shap_array[:, :, class_idx]).mean()
        class_shap_means.append(mean_shap)
else:
    class_shap_means = [np.abs(shap_array).mean()] * 5

colors_classes = plt.cm.RdYlGn_r(np.linspace(0.3, 0.7, 5))
ax1.bar(class_names, class_shap_means, color=colors_classes, edgecolor='black', linewidth=1.5)
ax1.set_ylabel('Mean |SHAP value|', fontsize=9)
ax1.set_title('Average SHAP Impact by Acuity', fontweight='bold', fontsize=10)
ax1.grid(axis='y', alpha=0.3)

# 2. Prediction Accuracy per Class (top-middle)
ax2 = fig.add_subplot(gs[0, 1])
from sklearn.metrics import precision_recall_fscore_support
pred_all = lgbm_acuity.predict(X_val_dense)
precision, recall, f1, _ = precision_recall_fscore_support(y_val_values, pred_all, average=None)
x_pos = np.arange(5)
width = 0.25
ax2.bar(x_pos - width, precision, width, label='Precision', alpha=0.8, color='steelblue')
ax2.bar(x_pos, recall, width, label='Recall', alpha=0.8, color='orange')
ax2.bar(x_pos + width, f1, width, label='F1-Score', alpha=0.8, color='green')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(class_names, fontsize=9)
ax2.set_ylabel('Score', fontsize=9)
ax2.set_title('Per-Class Performance', fontweight='bold', fontsize=10)
ax2.legend(fontsize=8)
ax2.set_ylim([0, 1.05])
ax2.grid(axis='y', alpha=0.3)

# 3. Feature Type Distribution (top-right)
ax3 = fig.add_subplot(gs[0, 2])
n_bert = sum(1 for f in feature_names_stage1 if f.startswith('clinical_bert_'))
n_chi2 = sum(1 for f in feature_names_stage1 if f.startswith('tfidf_'))
n_cat = sum(1 for f in feature_names_stage1 if f.startswith('cat_') or f.startswith('x0_'))
n_num = len(feature_names_stage1) - n_bert - n_chi2 - n_cat
feature_types = ['BERT\nEmbeddings', 'Clinical\nKeywords', 'Categorical', 'Numerical']
feature_counts = [n_bert, n_chi2, n_cat, n_num]
colors_features = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
wedges, texts, autotexts = ax3.pie(feature_counts, labels=feature_types, autopct='%1.1f%%',
                                     colors=colors_features, startangle=90, textprops={'fontsize': 8})
ax3.set_title('Feature Distribution', fontweight='bold', fontsize=10)

# 4. SHAP Value Distribution (middle - wide)
ax4 = fig.add_subplot(gs[1, :])
if len(shap_array.shape) == 3:
    for class_idx in range(5):
        class_shap_flat = shap_array[:, :, class_idx].flatten()
        ax4.hist(class_shap_flat, bins=50, alpha=0.5, label=class_names[class_idx], density=True)
else:
    shap_flat = shap_array.flatten()
    ax4.hist(shap_flat, bins=50, alpha=0.7, label='SHAP values', color='steelblue', density=True)

ax4.set_xlabel('SHAP value', fontsize=9)
ax4.set_ylabel('Density', fontsize=9)
ax4.set_title('SHAP Value Distribution Across All Features', fontweight='bold', fontsize=10)
ax4.legend(fontsize=8, loc='upper right')
ax4.grid(axis='y', alpha=0.3)

# 5. Confusion Matrix (bottom-left)
ax5 = fig.add_subplot(gs[2, 0])
cm_val = confusion_matrix(y_val_values, pred_all, labels=range(5))
im = ax5.imshow(cm_val, cmap='Blues', aspect='auto')
ax5.set_xticks(range(5))
ax5.set_yticks(range(5))
ax5.set_xticklabels(class_names, fontsize=8)
ax5.set_yticklabels(class_names, fontsize=8)
ax5.set_xlabel('Predicted', fontsize=9)
ax5.set_ylabel('Actual', fontsize=9)
ax5.set_title('Confusion Matrix (Validation)', fontweight='bold', fontsize=10)

for i in range(5):
    for j in range(5):
        text = ax5.text(j, i, cm_val[i, j], ha="center", va="center", 
                       color="white" if cm_val[i, j] > cm_val.max()/2 else "black",
                       fontsize=8, fontweight='bold')

plt.colorbar(im, ax=ax5, fraction=0.046, pad=0.04)

# 6. Top 10 Most Important Features (bottom-middle)
ax6 = fig.add_subplot(gs[2, 1])
if len(shap_array.shape) == 3:
    all_features_importance = np.abs(shap_array).mean(axis=(0, 2))
else:
    all_features_importance = np.abs(shap_array).mean(axis=0)

top_10_idx = np.argsort(all_features_importance)[-10:][::-1]
top_10_names = [feature_names_stage1[i][:28] + "..." if len(feature_names_stage1[i]) > 28 
                else feature_names_stage1[i] for i in top_10_idx]
top_10_scores = all_features_importance[top_10_idx]

ax6.barh(range(10), top_10_scores, color='steelblue', edgecolor='black', linewidth=1)
ax6.set_yticks(range(10))
ax6.set_yticklabels(top_10_names, fontsize=7)
ax6.set_xlabel('Mean |SHAP|', fontsize=9)
ax6.set_title('Top 10 Global Feature Importance', fontweight='bold', fontsize=10)
ax6.invert_yaxis()
ax6.grid(axis='x', alpha=0.3)

# 7. Model Statistics (bottom-right)
ax7 = fig.add_subplot(gs[2, 2])
ax7.axis('off')

stats_text = f"""
ACUITY MODEL STATISTICS
{'─'*42}
PERFORMANCE (Validation)
 Accuracy:           {acc:.4f} ({acc*100:.1f}%)
 Kappa (linear):     {kappa_linear:.4f}
 Kappa (quadratic):  {kappa_quadratic:.4f}
 F1 (weighted):      {f1_weighted:.4f}

CLINICAL SAFETY ⚠️
 Undertriage:        {undertriage_rate*100:.1f}%  (DANGER)
 Correct:            {correct_rate*100:.1f}%  (GOOD)
 Overtriage:         {overtriage_rate*100:.1f}%  (safe)

DATASET
 Training:           {X_train_dense.shape[0]:,}
 Validation:         {X_val_dense.shape[0]:,}
 Features:           {X_train_dense.shape[1]:,}
  • BERT:            {n_bert}
  • Keywords:        {n_chi2}
  • Categorical:     {n_cat}
  • Numerical:       {n_num}

TARGET: ESI Acuity (1-5)
"""

ax7.text(0.05, 0.95, stats_text, transform=ax7.transAxes, fontsize=8.5,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.4))

plt.savefig('acuity_model_dashboard.png', dpi=100, bbox_inches='tight')
plt.show()
print("\n✓ Saved: acuity_model_dashboard.png")

print(f"\n{'='*80}")
print("✅ ACUITY MODEL ANALYSIS COMPLETE!")
print(f"{'='*80}")
print(f"\nGenerated visualizations:")
print(f"  1. shap_dependence_plots.png          → Feature impact relationships")
print(f"  2. acuity_model_dashboard.png         → Complete statistics & metrics")
print(f"\n💡 These SHAP visualizations help clinicians understand")
print(f"   the model's decision-making process for acuity triage!")


In [ ]:

# ====================================================================
# FINAL COMPREHENSIVE ACUITY PREDICTION MODEL DASHBOARD
# ====================================================================
# Complete overview of model performance, interpretability, and clinical metrics
# Single-stage pipeline: Clinical Features + Chief Complaints → ESI Acuity (1-5)

print("\n" + "="*80)
print("📊 FINAL COMPREHENSIVE ACUITY MODEL ANALYSIS DASHBOARD")
print("="*80)

class_names = [f"ESI {i}" for i in range(1, 6)]

# Handle SHAP values format
if isinstance(shap_values, list):
    shap_array = np.array([sv for sv in shap_values])
else:
    shap_array = shap_values

# Create comprehensive final dashboard
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.35)

# 1. SHAP Distribution by Class (top-left)
ax1 = fig.add_subplot(gs[0, 0])
class_shap_means = []
if len(shap_array.shape) == 3:
    for class_idx in range(5):
        mean_shap = np.abs(shap_array[:, :, class_idx]).mean()
        class_shap_means.append(mean_shap)
else:
    class_shap_means = [np.abs(shap_array).mean()] * 5

colors_classes = plt.cm.RdYlGn_r(np.linspace(0.3, 0.7, 5))
ax1.bar(class_names, class_shap_means, color=colors_classes, edgecolor='black', linewidth=1.5)
ax1.set_ylabel('Mean |SHAP value|', fontsize=9)
ax1.set_title('SHAP Impact by Acuity Class', fontweight='bold', fontsize=10)
ax1.grid(axis='y', alpha=0.3)

# 2. Prediction Accuracy per Class (top-middle)
ax2 = fig.add_subplot(gs[0, 1])
from sklearn.metrics import precision_recall_fscore_support
pred_all = lgbm_acuity.predict(X_val_dense)
precision, recall, f1, _ = precision_recall_fscore_support(y_val_values, pred_all, average=None)
x_pos = np.arange(5)
width = 0.25
ax2.bar(x_pos - width, precision, width, label='Precision', alpha=0.8, color='steelblue')
ax2.bar(x_pos, recall, width, label='Recall', alpha=0.8, color='orange')
ax2.bar(x_pos + width, f1, width, label='F1-Score', alpha=0.8, color='green')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(class_names, fontsize=9)
ax2.set_ylabel('Score', fontsize=9)
ax2.set_title('Per-Class Performance Metrics', fontweight='bold', fontsize=10)
ax2.legend(fontsize=8)
ax2.set_ylim([0, 1.05])
ax2.grid(axis='y', alpha=0.3)

# 3. Feature Count per Type (top-right)
ax3 = fig.add_subplot(gs[0, 2])
n_bert = sum(1 for f in feature_names_stage1 if f.startswith('clinical_bert_'))
n_chi2 = sum(1 for f in feature_names_stage1 if f.startswith('tfidf_'))
n_cat = sum(1 for f in feature_names_stage1 if f.startswith('cat_') or f.startswith('x0_'))
n_num = len(feature_names_stage1) - n_bert - n_chi2 - n_cat
feature_types = ['ClinicalBERT', 'Chi2-TF-IDF', 'Categorical', 'Numerical']
feature_counts = [n_bert, n_chi2, n_cat, n_num]
colors_features = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
wedges, texts, autotexts = ax3.pie(feature_counts, labels=feature_types, autopct='%1.1f%%',
                                     colors=colors_features, startangle=90, textprops={'fontsize': 8})
ax3.set_title('Feature Distribution', fontweight='bold', fontsize=10)

# 4. SHAP Value Distribution (middle - wide)
ax4 = fig.add_subplot(gs[1, :])
if len(shap_array.shape) == 3:
    for class_idx in range(5):
        class_shap_flat = shap_array[:, :, class_idx].flatten()
        ax4.hist(class_shap_flat, bins=50, alpha=0.5, label=class_names[class_idx], density=True)
else:
    shap_flat = shap_array.flatten()
    ax4.hist(shap_flat, bins=50, alpha=0.7, label='SHAP values', color='steelblue', density=True)

ax4.set_xlabel('SHAP value', fontsize=9)
ax4.set_ylabel('Density', fontsize=9)
ax4.set_title('SHAP Value Distribution Across All Features and Classes', fontweight='bold', fontsize=10)
ax4.legend(fontsize=8, loc='upper right')
ax4.grid(axis='y', alpha=0.3)

# 5. Model Predictions vs Actual (Confusion Matrix - bottom-left)
ax5 = fig.add_subplot(gs[2, 0])
cm_val = confusion_matrix(y_val_values, pred_all, labels=range(5))
im = ax5.imshow(cm_val, cmap='Blues', aspect='auto')
ax5.set_xticks(range(5))
ax5.set_yticks(range(5))
ax5.set_xticklabels(class_names, fontsize=8)
ax5.set_yticklabels(class_names, fontsize=8)
ax5.set_xlabel('Predicted Acuity', fontsize=9)
ax5.set_ylabel('Actual Acuity', fontsize=9)
ax5.set_title('Confusion Matrix (Validation Set)', fontweight='bold', fontsize=10)

for i in range(5):
    for j in range(5):
        text = ax5.text(j, i, cm_val[i, j], ha="center", va="center", 
                       color="white" if cm_val[i, j] > cm_val.max()/2 else "black",
                       fontsize=8, fontweight='bold')

plt.colorbar(im, ax=ax5, fraction=0.046, pad=0.04)

# 6. Top 10 Most Important Features (bottom-middle)
ax6 = fig.add_subplot(gs[2, 1])
all_features_importance = np.abs(np.array(shap_array)).mean(axis=(0, 2)) if len(shap_array.shape) == 3 else np.abs(shap_array).mean(axis=0)
top_10_idx = np.argsort(all_features_importance)[-10:][::-1]
top_10_names = [feature_names_stage1[i][:30] + "..." if len(feature_names_stage1[i]) > 30 
                else feature_names_stage1[i] for i in top_10_idx]
top_10_scores = all_features_importance[top_10_idx]

ax6.barh(range(10), top_10_scores, color='steelblue', edgecolor='black', linewidth=1)
ax6.set_yticks(range(10))
ax6.set_yticklabels(top_10_names, fontsize=8)
ax6.set_xlabel('Mean |SHAP value|', fontsize=9)
ax6.set_title('Top 10 Global Feature Importance', fontweight='bold', fontsize=10)
ax6.invert_yaxis()
ax6.grid(axis='x', alpha=0.3)

# 7. Model Statistics (bottom-right)
ax7 = fig.add_subplot(gs[2, 2])
ax7.axis('off')

stats_text = f"""
ACUITY MODEL STATISTICS (Single-Stage)
{'─'*45}
PERFORMANCE (Validation Set)
 Accuracy:            {acc:.4f} ({acc*100:.2f}%)
 Cohen's Kappa (L):   {kappa_linear:.4f}
 Cohen's Kappa (Q):   {kappa_quadratic:.4f}
 F1 (macro):          {f1_macro:.4f}
 F1 (weighted):       {f1_weighted:.4f}

CLINICAL SAFETY METRICS ⚠️
 Undertriage Rate:    {undertriage_rate*100:.2f}% (DANGEROUS)
 Correct Rate:        {correct_rate*100:.2f}% (GOOD)
 Overtriage Rate:     {overtriage_rate*100:.2f}% (SAFE)

DATASET
 Training:            {X_train_dense.shape[0]:,} samples
 Validation:          {X_val_dense.shape[0]:,} samples
 Total Features:      {X_train_dense.shape[1]:,}
  • ClinicalBERT:     {n_bert} (768-dim)
  • Chi2 Keywords:    {n_chi2}
  • Categorical:      {n_cat}
  • Numerical:        {n_num}

TARGET: Triage Acuity (ESI 1-5)
EXPLAINABILITY: SHAP Values
"""

ax7.text(0.05, 0.95, stats_text, transform=ax7.transAxes, fontsize=8.5,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.35, pad=0.8))

plt.savefig('shap_comprehensive_final_dashboard.png', dpi=100, bbox_inches='tight')
plt.show()
print("\n✓ Saved: shap_comprehensive_final_dashboard.png")

print(f"\n{'='*80}")
print("✅ ACUITY PREDICTION PIPELINE COMPLETE!")
print(f"{'='*80}")
print(f"\n📌 PIPELINE SUMMARY:")
print(f"   Stage: Single-Stage Acuity Prediction")
print(f"   Target: ESI Acuity Level (1-5)")
print(f"   Model: LightGBM Multi-Class Classifier")
print(f"   Features: BERT embeddings + Clinical keywords + Vital signs + Demographics")
print(f"\n📊 Generated Visualizations:")
print(f"   • shap_dependence_plots.png          → Feature-Acuity relationships")
print(f"   • acuity_model_dashboard.png         → Key statistics & metrics")
print(f"   • shap_comprehensive_final_dashboard.png → Complete analysis dashboard")
print(f"\n💡 CLINICAL IMPACT:")
print(f"   ✓ Provides interpretable acuity predictions (ESI 1-5)")
print(f"   ✓ SHAP explanations help clinicians validate decisions")
print(f"   ✓ Identifies which features drive each acuity level")
print(f"   ✓ Highlights clinical safety metrics (undertriage vs overtriage)")


# SECTION 8: INTERACTIVE GRADIO INTERFACE FOR ACUITY PREDICTION

## Real-Time Acuity Prediction with Explainability

This section creates an interactive web interface using **Gradio** that allows users to:

1. **Input Clinical Data**: Enter vital signs, demographics, and patient information
2. **Add Chief Complaint**: Describe the patient's main complaint (free text)
3. **Get Instant Predictions**: Receive ESI acuity level (1-5) with confidence scores
4. **Understand the Decision**: View SHAP-based feature importance showing which clinical factors influenced the prediction

**How to Use**:
- Run this cell to launch the Gradio interface
- The interface will start a local web server (typically http://127.0.0.1:7860)
- Enter sample patient data and see real-time predictions
- Each prediction includes explainability via SHAP values

**Perfect for**:
- Clinical validation and testing
- Educational demonstrations
- Model debugging and refinement
- Stakeholder presentations



In [ ]:
import subprocess, sys
try:
    import gradio as gr
except:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gradio", "-q"])
    import gradio as gr

print("✓ Gradio ready\n")

# Extract features
all_features = [c for c in features_engineered_df.columns]
numerical_f = [c for c in all_features if c != 'chief_complaint_raw' and features_engineered_df[c].dtype in ['int64', 'float64'] 
               and not (features_engineered_df[c].nunique() <= 2 and features_engineered_df[c].min() >= 0 and features_engineered_df[c].max() <= 1)]
binary_f = [c for c in all_features if c != 'chief_complaint_raw' and features_engineered_df[c].dtype in ['int64', 'float64'] 
            and (features_engineered_df[c].nunique() <= 2 and features_engineered_df[c].min() >= 0 and features_engineered_df[c].max() <= 1)]
categorical_f = [c for c in all_features if c != 'chief_complaint_raw' and features_engineered_df[c].dtype == 'object']

print(f"✓ Features: {len(all_features)} | Numerical: {len(numerical_f)} | Binary: {len(binary_f)} | Categorical: {len(categorical_f)}\n")

def predict_fn(*inputs):
    try:
        d = {}
        idx = 0
        for col in numerical_f:
            d[col] = float(inputs[idx]) if idx < len(inputs) else features_engineered_df[col].median()
            idx += 1
        for col in categorical_f:
            d[col] = str(inputs[idx]) if idx < len(inputs) else ""
            idx += 1
        for col in binary_f:
            d[col] = int(inputs[idx]) if idx < len(inputs) else 0
            idx += 1
        d['chief_complaint_raw'] = str(inputs[idx]) if idx < len(inputs) else "routine visit"
        
        df_in = pd.DataFrame([d])
        for col in features_engineered_df.columns:
            if col not in df_in.columns:
                df_in[col] = features_engineered_df[col].median() if features_engineered_df[col].dtype in ['int64', 'float64'] else 0
        
        df_in = df_in[features_engineered_df.columns]
        df_eng = engineer_features(df_in.copy())
        X = preprocessor.transform(df_eng)
        X_dense = X.toarray() if hasattr(X, 'toarray') else X
        
        pred = int(lgbm_acuity.predict(X_dense)[0]) - 1
        probs = lgbm_acuity.predict_proba(X_dense)[0]
        
        names = ["ESI 1: Immediate", "ESI 2: Urgent", "ESI 3: Semi-Urgent", "ESI 4: Minor", "ESI 5: Non-Urgent"]
        
        # Prob chart
        f, ax = plt.subplots(figsize=(12, 5))
        c = ['#d32f2f', '#f57c00', '#fbc02d', '#7cb342', '#388e3c']
        b = ax.barh(range(5), probs * 100, color=c, edgecolor='black', linewidth=2.5)
        b[pred].set_linewidth(4)
        b[pred].set_edgecolor('gold')
        ax.set_yticks(range(5))
        ax.set_yticklabels([f"ESI {i+1}" for i in range(5)], fontsize=11, fontweight='bold')
        ax.set_xlabel('Probability (%)', fontsize=11, fontweight='bold')
        ax.set_title(f"PREDICTED: {names[pred]} | Confidence: {max(probs)*100:.1f}%", fontsize=12, fontweight='bold', color='darkgreen')
        ax.set_xlim([0, 100])
        ax.grid(axis='x', alpha=0.3)
        for i, (bar, p) in enumerate(zip(b, probs)):
            ax.text(p*100 + 2, i, f'{p*100:.1f}%', va='center', fontweight='bold')
        plt.tight_layout()
        
        # SHAP
        f_shap = None
        try:
            ex = shap.TreeExplainer(lgbm_acuity)
            sv = ex.shap_values(X_dense)
            if isinstance(sv, list):
                sv_p = sv[pred][0]
            else:
                sv_p = sv[0, :, pred] if len(sv.shape) == 3 else sv[0]
            
            f_shap, ax_s = plt.subplots(figsize=(12, 6))
            ti = np.argsort(np.abs(sv_p))[-12:][::-1]
            tn = [feature_names_stage1[i][:40] if i < len(feature_names_stage1) else f"F{i}" for i in ti]
            tv = sv_p[ti]
            
            cs = ['#c62828' if v < 0 else '#00796b' for v in tv]
            ax_s.barh(range(len(tv)), tv, color=cs, alpha=0.8, edgecolor='black')
            ax_s.set_yticks(range(len(tv)))
            ax_s.set_yticklabels(tn, fontsize=9)
            ax_s.set_xlabel('SHAP Value', fontsize=11, fontweight='bold')
            ax_s.axvline(x=0, color='black', linestyle='-', linewidth=2)
            ax_s.set_title(f"Why {names[pred]}? (Top 12)", fontsize=12, fontweight='bold')
            ax_s.invert_yaxis()
            ax_s.grid(axis='x', alpha=0.3)
            plt.tight_layout()
        except:
            pass
        
        # Summary
        summ = f"""
╔════════════════════════════════════════════════════════════════════╗
║  🏥 ACUITY PREDICTION SUMMARY                                     ║
╠════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  PREDICTED: {names[pred]:<43}║
║  CONFIDENCE: {max(probs)*100:>6.1f}%{' '*43}║
║                                                                    ║
║  DISTRIBUTION:                                                    ║
"""
        for i in range(5):
            bar_len = int(probs[i] * 20)
            summ += f"║  ESI {i+1}: {probs[i]*100:>5.1f}%  {'█'*bar_len:<20}║\n"
        
        summ += """║                                                                    ║
╚════════════════════════════════════════════════════════════════════╝
"""
        return summ, f, f_shap
    
    except Exception as e:
        return f"Error: {str(e)}", None, None

# Create interface
print("🎨 Building interface...")

inp = []
for col in numerical_f:
    data = features_engineered_df[col].dropna()
    if len(data) > 0:
        inp.append(gr.Slider(minimum=float(data.min()), maximum=float(data.max()), value=float(data.median()), label=col.replace('_', ' ').title()))

for col in categorical_f:
    uniq = list(features_engineered_df[col].dropna().unique())
    if len(uniq) > 0:
        inp.append(gr.Dropdown(choices=uniq, value=uniq[0], label=col.replace('_', ' ').title()))

for col in binary_f:
    inp.append(gr.Checkbox(value=False, label=col.replace('_', ' ').title()))

inp.append(gr.Textbox(label="Chief Complaint", placeholder="e.g., 'Chest pain'", lines=2))

print(f"✓ {len(inp)} input components\n")

with gr.Blocks(title="ED Acuity Prediction", analytics_enabled=False) as demo:
    gr.Markdown("# 🏥 ED Acuity Prediction System")
    gr.Markdown("Predict ESI triage acuity (1-5) with SHAP explainability")
    
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Inputs")
            for c in inp:
                c.render()
            btn = gr.Button("Predict", variant="primary")
        
        with gr.Column(scale=1):
            gr.Markdown("### Results")
            out_txt = gr.Textbox(label="Summary", lines=10)
            out_prob = gr.Plot(label="Probabilities")
            out_shap = gr.Plot(label="SHAP")
    
    btn.click(predict_fn, inputs=inp, outputs=[out_txt, out_prob, out_shap])

print("🌐 Launching at http://127.0.0.1:7860\n")
demo.launch(share=True)


In [ ]:
# # load fedmml_ed_triage_dataset_final.csv, preprocess it and predict triage_acuity
# fedmml_df = pd.read_csv('dataset/fedmml_ed_triage_dataset_final.csv')
# print("✓ Dataset loaded\n")
# # Preprocess
# fedmml_df = engineer_features(fedmml_df)
# X,y = fedmml_df.drop(columns=['triage_acuity']), fedmml_df['triage_acuity']
# X_pp= preprocessor.transform(X)
# print("✓ Dataset preprocessed\n")
# # Predict
# preds = lgbm_acuity.predict(X_pp)
# print("✓ Predictions made\n")
# # Show distribution of predicted triage acuity
# import matplotlib.pyplot as plt
# plt.hist(preds, bins=np.arange(1, 7)-0.5, edgecolor='black')
# plt.xticks(range(1, 6))
# plt.xlabel('Predicted Triage Acuity (ESI 1-5)')
# plt.ylabel('Count')
# plt.title('Distribution of Predicted Triage Acuity')
# plt.grid(axis='y', alpha=0.3)
# plt.show()

# # print statistics, classification report, kappa cohen (already imported)
# print(classification_report(y, preds))
# print(f"Cohen's Kappa (Linear): {cohen_kappa_score(y, preds, weights='linear'):.4f}")
# print(f"Cohen's Kappa (Quadratic): {cohen_kappa_score(y, preds, weights='quadratic'):.4f}")
